In [ ]:
#@title Cell 0 — Setup (run me first)
# ============================================================
# Shared Infrastructure: company_sim.py (embedded)
# ============================================================
import json, os
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

COLORS = {
    'ols': '#2E5EA8',
    'truth': '#D4A017',
    'bias': '#C0392B',
    'repair': '#27AE60',
    'alt': '#7F8C8D',
}

_trap_responses = {}
_TRAP_LOG_PATH = '/content/trap_log.json'

def _load_trap_log():
    global _trap_responses
    if os.path.exists(_TRAP_LOG_PATH):
        try:
            with open(_TRAP_LOG_PATH, 'r') as f:
                _trap_responses = json.load(f)
        except (json.JSONDecodeError, IOError):
            _trap_responses = {}

def _save_trap_log():
    try:
        os.makedirs(os.path.dirname(_TRAP_LOG_PATH), exist_ok=True)
        with open(_TRAP_LOG_PATH, 'w') as f:
            json.dump(_trap_responses, f, indent=2)
    except IOError:
        pass

def record_trap_response(notebook_id, question_id, response):
    key = f"{notebook_id}_{question_id}"
    _trap_responses[key] = {"response": response, "timestamp": datetime.now().isoformat()}
    _save_trap_log()

def get_trap_response(notebook_id, question_id):
    key = f"{notebook_id}_{question_id}"
    entry = _trap_responses.get(key)
    return entry["response"] if entry else None

def check_gate(notebook_id, question_id):
    return f"{notebook_id}_{question_id}" in _trap_responses

def get_all_responses():
    return dict(_trap_responses)

def create_trap_widget(notebook_id, question_id, question_text, options):
    label = widgets.HTML(f"<b>{question_text}</b>")
    radio = widgets.RadioButtons(options=options, value=None, layout=widgets.Layout(width='auto'))
    submit = widgets.Button(description="Submit", button_style='primary')
    output = widgets.Output()
    def on_submit(btn):
        with output:
            output.clear_output()
            if radio.value is None:
                print("Please select an option before submitting.")
                return
            record_trap_response(notebook_id, question_id, radio.value)
            print(f"Response recorded: {radio.value}")
            submit.disabled = True
            radio.disabled = True
    submit.on_click(on_submit)
    existing = get_trap_response(notebook_id, question_id)
    if existing is not None:
        try:
            radio.value = existing
            radio.disabled = True
            submit.disabled = True
        except Exception:
            pass
    return widgets.VBox([label, radio, submit, output])

_load_trap_log()

class CompanySimulator:
    def __init__(self, endogeneity_strength=20, heteroscedasticity_strength=0.5,
                 nonlinearity=True, bad_control_strength=0.1, noise_sigma=1.0):
        self.endogeneity_strength = endogeneity_strength
        self.heteroscedasticity_strength = heteroscedasticity_strength
        self.nonlinearity = nonlinearity
        self.bad_control_strength = bad_control_strength
        self.noise_sigma = noise_sigma
        self.beta_0, self.beta_1, self.beta_2, self.beta_U = 50, 8, 3, 2
        self.staff_loading = 5
    def generate(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1, eta2, eta3 = [rng.normal(0, self.noise_sigma, n) for _ in range(3)]
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity: X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps = rng.normal(0, np.sqrt(np.abs(self.heteroscedasticity_strength * X1)))
        Y = self.beta_0 + (self.beta_1 * np.log(X1) if self.nonlinearity else self.beta_1 * X1) + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        return pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2, 'satisfaction': X3})
    def truth(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1, eta2, eta3 = [rng.normal(0, self.noise_sigma, n) for _ in range(3)]
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity: X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps = rng.normal(0, np.sqrt(np.abs(self.heteroscedasticity_strength * X1)))
        Y = self.beta_0 + (self.beta_1 * np.log(X1) if self.nonlinearity else self.beta_1 * X1) + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        df = pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2, 'satisfaction': X3, 'demand_U': U})
        return df, {'beta_0': self.beta_0, 'beta_1': self.beta_1, 'beta_2': self.beta_2, 'beta_U': self.beta_U}
    def set_heteroscedasticity(self, s): self.heteroscedasticity_strength = s
    def set_endogeneity(self, s): self.endogeneity_strength = s
    def set_nonlinearity(self, c): self.nonlinearity = bool(c)

class MonteCarloEngine:
    def run(self, estimator_fn, param_name, param_grid, simulator, n_reps=5000, n_obs=500):
        results = np.empty((len(param_grid), n_reps))
        setter = getattr(simulator, f'set_{param_name}', None)
        for i, val in enumerate(param_grid):
            if setter: setter(val)
            for r in range(n_reps):
                results[i, r] = estimator_fn(simulator.generate(n=n_obs, seed=r))
        return results
    def quick_run(self, estimator_fn, dgp_fn, n_reps=1000, n_obs=500):
        return np.array([estimator_fn(dgp_fn(seed=r, n=n_obs)) for r in range(n_reps)])

class DiagnosticSuite:
    @staticmethod
    def run_diagnostics(model_result):
        fig, axes = plt.subplots(2, 2, figsize=(10, 8))
        influence = OLSInfluence(model_result)
        fitted = model_result.fittedvalues
        resid = model_result.resid
        std_resid = influence.resid_studentized_internal
        ax = axes[0, 0]
        ax.scatter(fitted, resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1.5)
        ax.set_xlabel('Fitted values'); ax.set_ylabel('Residuals'); ax.set_title('Residuals vs Fitted')
        ax = axes[0, 1]
        stats.probplot(resid, dist='norm', plot=ax)
        ax.set_title('Normal Q-Q'); ax.get_lines()[0].set_color(COLORS['ols']); ax.get_lines()[1].set_color(COLORS['bias'])
        ax = axes[1, 0]
        ax.scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.4, s=15, color=COLORS['ols'])
        ax.set_xlabel('Fitted values'); ax.set_ylabel(r'$\sqrt{|Standardized\ Residuals|}$'); ax.set_title('Scale-Location')
        ax = axes[1, 1]
        leverage = influence.hat_matrix_diag
        ax.scatter(leverage, std_resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1)
        ax.set_xlabel('Leverage'); ax.set_ylabel('Standardized Residuals'); ax.set_title('Residuals vs Leverage')
        x_range = np.linspace(0.001, ax.get_xlim()[1], 100)
        for cook_val in [0.5, 1.0]:
            for sign in [1, -1]:
                p = model_result.df_model + 1
                y_val = sign * np.sqrt(cook_val * p * (1 - x_range) / x_range)
                ax.plot(x_range, y_val, '--', color=COLORS['bias'], alpha=0.5,
                        label=f"Cook's d={cook_val}" if sign == 1 else None)
        ax.legend(fontsize=8)
        fig.tight_layout()
        return fig

    @staticmethod
    def summary_tests(model_result):
        results = {}
        exog = model_result.model.exog
        exog_names = model_result.model.exog_names
        vif_dict = {}
        for i, name in enumerate(exog_names):
            if name == 'const': continue
            vif_dict[name] = variance_inflation_factor(exog, i)
        results['vif'] = vif_dict
        bp_stat, bp_pval, _, _ = het_breuschpagan(model_result.resid, model_result.model.exog)
        results['breusch_pagan'] = (bp_stat, bp_pval)
        results['durbin_watson'] = durbin_watson(model_result.resid)
        jb_stat, jb_pval, _, _ = jarque_bera(model_result.resid)
        results['jarque_bera'] = (jb_stat, jb_pval)
        return results

class AutopsyReport:
    @staticmethod
    def lightweight(notebook_number):
        threat = widgets.Text(description='Biggest threat:', placeholder='What is the biggest threat to this estimate?',
                              layout=widgets.Layout(width='90%'), style={'description_width': '120px'})
        check = widgets.Text(description='How to check:', placeholder='How would you check if that threat is real?',
                             layout=widgets.Layout(width='90%'), style={'description_width': '120px'})
        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()
        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                record_trap_response(nb_id, 'threat', threat.value)
                record_trap_response(nb_id, 'check', check.value)
                print('Autopsy responses saved.')
                submit.disabled = True
        submit.on_click(on_submit)
        return widgets.VBox([widgets.HTML(f'<h3>Mini Autopsy \u2014 Notebook {notebook_number}</h3>'), threat, check, submit, output])

    @staticmethod
    def full(notebook_number, available_sidebars=None):
        fields = {
            'point_estimate': widgets.Text(description='Point estimate:', placeholder='My point estimate is:',
                                           layout=widgets.Layout(width='90%'), style={'description_width': '150px'}),
            'robustness': widgets.Text(description='Robustness value:', placeholder='The robustness value is:',
                                       layout=widgets.Layout(width='90%'), style={'description_width': '150px'}),
            'partial_r2': widgets.Text(description='Strongest partial R\u00b2:', placeholder='The strongest observed covariate has partial R\u00b2 of:',
                                       layout=widgets.Layout(width='90%'), style={'description_width': '150px'}),
            'plain_language': widgets.Text(description='Plain language:', placeholder='An omitted variable would need to be ___ times as important as ___ to explain away this result',
                                           layout=widgets.Layout(width='90%'), style={'description_width': '150px'}),
        }
        children = [widgets.HTML(f'<h3>Full Autopsy Report \u2014 Notebook {notebook_number}</h3>')]
        children.extend(fields.values())
        if available_sidebars:
            sidebar_dropdown = widgets.Dropdown(options=['(select)'] + list(available_sidebars),
                                                description='Most analogous disaster:', layout=widgets.Layout(width='90%'),
                                                style={'description_width': '180px'})
            children.append(sidebar_dropdown)
            fields['analogous_disaster'] = sidebar_dropdown
        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()
        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                for key, w in fields.items():
                    record_trap_response(nb_id, key, w.value)
                print('Full autopsy report saved.')
                submit.disabled = True
        submit.on_click(on_submit)
        children.extend([submit, output])
        return widgets.VBox(children)

# ============================================================
# Generate Regression Discontinuity Data
# ============================================================
def generate_rdd_data(seed=42):
    rng = np.random.default_rng(seed)
    n = 1000
    TRUE_EFFECT = 5000  # +$5,000 scholarship effect on earnings
    CUTOFF = 0
    
    # Running variable: exam score centered at cutoff
    score = rng.uniform(-50, 50, n)
    
    # Treatment: scholarship if score >= cutoff
    treatment = (score >= CUTOFF).astype(float)
    
    # Demographics (smooth across cutoff by construction)
    age = rng.normal(22, 3, n).clip(18, 35)
    parent_income = rng.normal(50000, 15000, n).clip(10000, 120000)
    gpa_prior = rng.normal(3.0, 0.5, n).clip(1.5, 4.0)
    female = rng.binomial(1, 0.52, n)
    
    # Potential outcome: smooth function of score + treatment jump
    # Y = f(score) + treatment_effect * D + noise
    # f(score) is smooth and continuous at cutoff
    baseline = (35000 + 150 * score + 0.8 * score**2  # smooth in score
                + 500 * gpa_prior + 0.05 * parent_income  # covariates
                + rng.normal(0, 3000, n))  # noise
    
    earnings = baseline + TRUE_EFFECT * treatment
    
    data = pd.DataFrame({
        'score': score,
        'treatment': treatment,
        'earnings': earnings,
        'age': age,
        'parent_income': parent_income,
        'gpa_prior': gpa_prior,
        'female': female,
    })
    
    return data, TRUE_EFFECT, CUTOFF

rdd_data, TRUE_RDD_EFFECT, CUTOFF = generate_rdd_data(seed=42)

print('\u2705 Setup complete.')
print(f'RDD data: {len(rdd_data)} students')
print(f'Cutoff: score = {CUTOFF} (scholarship if score \u2265 {CUTOFF})')
print(f'{int(rdd_data.treatment.sum())} received scholarships, {int((1-rdd_data.treatment).sum())} did not')

# Notebook 8: The Redemption

**The finale.**

---

## The Scholarship Cutoff

A university awards merit scholarships based on an entrance exam. Students who score **at or above the cutoff** receive a full scholarship. Students just below get nothing.

Think about the students near the cutoff. The student who scored 0.1 points above and the student who scored 0.1 points below are virtually identical — in ability, preparation, motivation, family background, everything. The tiny difference in score is essentially **random noise**.

Yet one got a scholarship and the other didn't.

This is a **regression discontinuity design**. Near the cutoff, treatment is essentially random. And that means we can estimate the causal effect of the scholarship without worrying about all the confounders that plagued Notebooks 1–7.

In [ ]:
# ============================================================
# The Visual: The most important plot in the course
# ============================================================

fig, ax = plt.subplots(figsize=(12, 7))

# Scatter: all points
below = rdd_data[rdd_data['score'] < CUTOFF]
above = rdd_data[rdd_data['score'] >= CUTOFF]

ax.scatter(below['score'], below['earnings'], alpha=0.3, s=20,
           color=COLORS['ols'], marker='o', label='No scholarship')
ax.scatter(above['score'], above['earnings'], alpha=0.3, s=20,
           color=COLORS['repair'], marker='s', label='Scholarship')

# Fit separate regression lines on each side
# Left side
X_left = sm.add_constant(below['score'])
model_left = sm.OLS(below['earnings'], X_left).fit()
score_left = np.linspace(below['score'].min(), CUTOFF - 0.01, 100)
pred_left = model_left.predict(sm.add_constant(score_left))
ax.plot(score_left, pred_left, color=COLORS['ols'], linewidth=2.5, linestyle='-')

# Right side
X_right = sm.add_constant(above['score'])
model_right = sm.OLS(above['earnings'], X_right).fit()
score_right = np.linspace(CUTOFF, above['score'].max(), 100)
pred_right = model_right.predict(sm.add_constant(score_right))
ax.plot(score_right, pred_right, color=COLORS['repair'], linewidth=2.5, linestyle='-')

# Vertical cutoff line
ax.axvline(CUTOFF, color=COLORS['truth'], linewidth=2, linestyle='--',
           label='Cutoff', alpha=0.8)

# The GAP annotation
pred_at_cutoff_left = model_left.predict(sm.add_constant(np.array([CUTOFF])))[0]
pred_at_cutoff_right = model_right.predict(sm.add_constant(np.array([CUTOFF])))[0]
gap = pred_at_cutoff_right - pred_at_cutoff_left

ax.annotate('', xy=(CUTOFF + 1, pred_at_cutoff_right),
            xytext=(CUTOFF + 1, pred_at_cutoff_left),
            arrowprops=dict(arrowstyle='<->', color=COLORS['truth'], lw=2.5))
ax.text(CUTOFF + 3, (pred_at_cutoff_left + pred_at_cutoff_right) / 2,
        f'Gap = ${gap:,.0f}\n(causal effect)',
        fontsize=13, fontweight='bold', color=COLORS['truth'],
        verticalalignment='center')

ax.set_xlabel('Exam Score (centered at cutoff)', fontsize=13)
ax.set_ylabel('Future Earnings ($)', fontsize=13)
ax.set_title(f'Regression Discontinuity: Scholarship Effect = ${gap:,.0f}',
             fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.tick_params(labelsize=11)

fig.tight_layout()
plt.show()

print(f'\nEstimated causal effect of scholarship: ${gap:,.0f}')
print(f'True effect: ${TRUE_RDD_EFFECT:,.0f}')
print(f'\nThe gap at the cutoff IS the causal effect.')
print('Visible. Credible. Beautiful.')

In [ ]:
# ============================================================
# Why This Works: Covariate Balance
# ============================================================

print('WHY THIS WORKS: Near the cutoff, treatment is essentially random.')
print('='*70)
print()
print('If the design is valid, students just above and just below the cutoff')
print('should look nearly identical on all pre-treatment characteristics.')
print('This is what makes RDD credible \u2014 the unobservables that plagued')
print('Notebooks 1\u20137 are balanced BY DESIGN.')
print()

# Narrow bandwidth for balance check
bw = 5  # Within 5 points of cutoff
near_above = rdd_data[(rdd_data['score'] >= CUTOFF) & (rdd_data['score'] < CUTOFF + bw)]
near_below = rdd_data[(rdd_data['score'] < CUTOFF) & (rdd_data['score'] >= CUTOFF - bw)]

balance_vars = ['age', 'parent_income', 'gpa_prior', 'female']
balance_labels = ['Age', 'Parent Income', 'Prior GPA', '% Female']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, (var, label) in enumerate(zip(balance_vars, balance_labels)):
    ax = axes[idx // 2, idx % 2]
    
    vals_below = near_below[var]
    vals_above = near_above[var]
    
    bins = np.linspace(min(vals_below.min(), vals_above.min()),
                       max(vals_below.max(), vals_above.max()), 20)
    ax.hist(vals_below, bins=bins, alpha=0.5, density=True,
            color=COLORS['ols'], label=f'Just below (n={len(near_below)})', edgecolor='white')
    ax.hist(vals_above, bins=bins, alpha=0.5, density=True,
            color=COLORS['repair'], label=f'Just above (n={len(near_above)})', edgecolor='white')
    
    # t-test
    t_stat, p_val = stats.ttest_ind(vals_below, vals_above)
    status = '\u2705' if p_val > 0.05 else '\u274c'
    ax.set_title(f'{label}: p = {p_val:.3f} {status}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylabel('Density')

fig.suptitle(f'Covariate Balance: Students within {bw} points of cutoff',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

print(f'\nBalance check (within {bw} points of cutoff):')
print(f'{"Variable":>15s}  {"Below":>10s}  {"Above":>10s}  {"Diff":>10s}  {"p-value":>8s}')
print('-' * 60)
for var, label in zip(balance_vars, balance_labels):
    m_below = near_below[var].mean()
    m_above = near_above[var].mean()
    _, p = stats.ttest_ind(near_below[var], near_above[var])
    status = '\u2705' if p > 0.05 else '\u274c'
    fmt = '.0f' if var == 'parent_income' else '.2f'
    print(f'{label:>15s}  {m_below:>10{fmt}}  {m_above:>10{fmt}}  {m_above-m_below:>+10{fmt}}  {p:>8.3f} {status}')

print()
print('\u27a1\ufe0f Groups are nearly identical near the cutoff.')
print('  This is what makes regression discontinuity credible.')

In [ ]:
# ============================================================
# The Bandwidth Tradeoff (interactive)
# ============================================================

out_bw = widgets.Output()

def update_bandwidth(bandwidth):
    with out_bw:
        clear_output(wait=True)
        
        # Subset data within bandwidth
        mask = (rdd_data['score'] >= CUTOFF - bandwidth) & (rdd_data['score'] <= CUTOFF + bandwidth)
        sub = rdd_data[mask].copy()
        
        below_sub = sub[sub['score'] < CUTOFF]
        above_sub = sub[sub['score'] >= CUTOFF]
        
        if len(below_sub) < 5 or len(above_sub) < 5:
            print('Too few observations. Increase bandwidth.')
            return
        
        # Fit local linear regression
        X_l = sm.add_constant(below_sub['score'])
        X_r = sm.add_constant(above_sub['score'])
        m_l = sm.OLS(below_sub['earnings'], X_l).fit()
        m_r = sm.OLS(above_sub['earnings'], X_r).fit()
        
        # Estimate at cutoff
        pred_l = m_l.predict(sm.add_constant(np.array([CUTOFF])))[0]
        pred_r = m_r.predict(sm.add_constant(np.array([CUTOFF])))[0]
        effect = pred_r - pred_l
        
        # SE via delta method approximation
        se = np.sqrt(m_l.bse['const']**2 + m_r.bse['const']**2)
        
        fig, ax = plt.subplots(figsize=(12, 7))
        
        # Gray out-of-bandwidth points
        out_mask = ~mask
        if out_mask.sum() > 0:
            ax.scatter(rdd_data.loc[out_mask, 'score'], rdd_data.loc[out_mask, 'earnings'],
                       alpha=0.05, s=10, color='gray')
        
        # In-bandwidth points
        ax.scatter(below_sub['score'], below_sub['earnings'], alpha=0.4, s=20,
                   color=COLORS['ols'], marker='o')
        ax.scatter(above_sub['score'], above_sub['earnings'], alpha=0.4, s=20,
                   color=COLORS['repair'], marker='s')
        
        # Regression lines within bandwidth
        sc_l = np.linspace(CUTOFF - bandwidth, CUTOFF - 0.01, 100)
        sc_r = np.linspace(CUTOFF, CUTOFF + bandwidth, 100)
        ax.plot(sc_l, m_l.predict(sm.add_constant(sc_l)), color=COLORS['ols'], linewidth=2.5)
        ax.plot(sc_r, m_r.predict(sm.add_constant(sc_r)), color=COLORS['repair'], linewidth=2.5)
        
        # Cutoff and bandwidth markers
        ax.axvline(CUTOFF, color=COLORS['truth'], linewidth=2, linestyle='--', alpha=0.8)
        ax.axvline(CUTOFF - bandwidth, color='gray', linewidth=1, linestyle=':', alpha=0.5)
        ax.axvline(CUTOFF + bandwidth, color='gray', linewidth=1, linestyle=':', alpha=0.5)
        
        # Gap annotation
        ax.annotate('', xy=(CUTOFF + 0.5, pred_r), xytext=(CUTOFF + 0.5, pred_l),
                    arrowprops=dict(arrowstyle='<->', color=COLORS['truth'], lw=2))
        
        ax.set_xlabel('Exam Score', fontsize=13)
        ax.set_ylabel('Earnings ($)', fontsize=13)
        ax.set_title(f'Bandwidth = {bandwidth:.0f}. Estimated effect = ${effect:,.0f}. SE = ${se:,.0f}.',
                     fontsize=14, fontweight='bold')
        ax.tick_params(labelsize=11)
        
        fig.tight_layout()
        plt.show()
        
        bias = abs(effect - TRUE_RDD_EFFECT)
        print(f'Bandwidth: {bandwidth:.0f} | Observations: {len(sub)} | '
              f'Effect: ${effect:,.0f} | SE: ${se:,.0f} | '
              f'True effect: ${TRUE_RDD_EFFECT:,.0f} | Bias: ${bias:,.0f}')
        
        if bandwidth <= 8:
            print('\u27a1\ufe0f Narrow bandwidth: less bias but more variance (lines wobble).')
        elif bandwidth >= 35:
            print('\u27a1\ufe0f Wide bandwidth: less variance but more bias (far-from-cutoff obs not comparable).')
        else:
            print('\u27a1\ufe0f Sweet spot: balancing bias and variance.')

slider_bw = widgets.IntSlider(value=15, min=3, max=50, step=1,
                               description='Bandwidth:',
                               style={'description_width': 'initial'},
                               layout=widgets.Layout(width='60%'))

print('THE BANDWIDTH TRADEOFF')
print('='*70)
print('Narrow bandwidth = low bias, high variance (lines wobble).')
print('Wide bandwidth = low variance, high bias (non-comparable obs included).')
print('Find the sweet spot.\n')
display(slider_bw, out_bw)

slider_bw.observe(lambda c: update_bandwidth(c['new']), names='value')
update_bandwidth(slider_bw.value)

In [ ]:
# ============================================================
# The Final Autopsy + Side-by-Side Comparison
# ============================================================

print('THE FINAL AUTOPSY')
print('='*70)
print()

# RDD model for autopsy
bw_opt = 15
mask_opt = (rdd_data['score'] >= CUTOFF - bw_opt) & (rdd_data['score'] <= CUTOFF + bw_opt)
sub_opt = rdd_data[mask_opt]
X_rdd = sm.add_constant(sub_opt[['score', 'treatment']])
model_rdd = sm.OLS(sub_opt['earnings'], X_rdd).fit()
rdd_coef = model_rdd.params['treatment']
rdd_se = model_rdd.bse['treatment']
rdd_t = rdd_coef / rdd_se
rdd_rv = abs(rdd_t) / (abs(rdd_t) + np.sqrt(model_rdd.df_resid))

print(f'RDD estimate: ${rdd_coef:,.0f} (SE: ${rdd_se:,.0f})')
print(f'Robustness value: {rdd_rv:.3f}')
print(f't-statistic: {rdd_t:.2f}')
print()

# Side-by-side comparison: NB1 vs NB8
print('='*70)
print('SIDE-BY-SIDE: Notebook 1 (NovaMart) vs. Notebook 8 (RDD)')
print('='*70)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NB1 report (red, fragile)
ax = axes[0]
nb1_items = [
    ('Coefficient bias', 'HIGH', COLORS['bias']),
    ('Selection bias', 'SEVERE', COLORS['bias']),
    ('Confounding', 'UNCONTROLLED', COLORS['bias']),
    ('Robustness', 'LOW', COLORS['bias']),
    ('Causal claim', 'UNSUPPORTED', COLORS['bias']),
]
for i, (label, status, color) in enumerate(nb1_items):
    ax.barh(i, 1, color=color, alpha=0.3, edgecolor=color)
    ax.text(0.5, i, f'{label}: {status}', ha='center', va='center',
            fontsize=11, fontweight='bold', color=color)
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title('NB1: Observational OLS\n(Fragile)', fontsize=14,
             fontweight='bold', color=COLORS['bias'])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)

# NB8 report (green, credible)
ax = axes[1]
nb8_items = [
    ('Coefficient bias', 'MINIMAL', COLORS['repair']),
    ('Selection bias', 'ELIMINATED', COLORS['repair']),
    ('Confounding', 'BALANCED BY DESIGN', COLORS['repair']),
    ('Robustness', f'RV = {rdd_rv:.3f}', COLORS['repair']),
    ('Causal claim', 'CREDIBLE', COLORS['repair']),
]
for i, (label, status, color) in enumerate(nb8_items):
    ax.barh(i, 1, color=color, alpha=0.3, edgecolor=color)
    ax.text(0.5, i, f'{label}: {status}', ha='center', va='center',
            fontsize=11, fontweight='bold', color=color)
ax.set_xlim(0, 1)
ax.set_yticks([])
ax.set_xticks([])
ax.set_title('NB8: Regression Discontinuity\n(Credible)', fontsize=14,
             fontweight='bold', color=COLORS['repair'])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)

fig.suptitle('Same tool. Same estimator. Same code. Different credibility.',
             fontsize=15, fontweight='bold', y=1.05)
fig.tight_layout()
plt.show()

print()
print('Same tool. Same estimator. Same code. Different credibility.')
print("The difference isn't the regression. The difference is the DESIGN.")
print()

# Full autopsy widget
autopsy = AutopsyReport.full(
    notebook_number=8,
    available_sidebars=[
        'NB1: Hormone therapy that killed',
        'NB2: Google Flu Trends',
        'NB3: Genome-wide association studies',
        'NB4: Challenger disaster',
        'NB5: Long-Term Capital Management',
        'NB6: Police and crime',
        'NB7: Hormone therapy (expanded)',
    ]
)
display(autopsy)

In [ ]:
# ============================================================
# Misconception Dashboard: All trap responses
# ============================================================

print('MISCONCEPTION DASHBOARD')
print('='*70)
print('Your journey through the Regression Autopsy course.\n')

# Define the expected trap questions and correct answers
trap_map = [
    ('notebook_1', 'q1', 'NB1: Coefficient',
     'Based on this regression, what do you conclude?',
     'The coefficient is biased upward \u2014 the true effect is smaller.',
     'Omitted Variable Bias'),
    ('notebook_2', 'q1', 'NB2: Uncertainty',
     'Are you confident the true effect is in this CI?',
     'No \u2014 heteroscedasticity invalidates classical SEs.',
     'Variance Decomposition'),
    ('notebook_3', 'q1', 'NB3: Significance',
     'Which variables have a meaningful effect?',
     'Statistical significance \u2260 practical significance.',
     't-Statistic Anatomy'),
    ('notebook_4', 'q1', 'NB4: Model',
     'How much revenue would a $2M campaign generate?',
     'Linear extrapolation overshoots; true form is log.',
     'Specification Error'),
    ('notebook_5', 'q1', 'NB5: R\u00b2',
     'Which model wins?',
     'Highest training R\u00b2 overfits; test R\u00b2 is negative.',
     'Bias-Variance Tradeoff'),
    ('notebook_6', 'q1', 'NB6: Causation',
     'Does increasing ad spend cause revenue to increase?',
     'OLS cannot determine direction of causality.',
     'IV Consistency'),
    ('notebook_7', 'q1', 'NB7: Real World',
     'Does the job training program work?',
     "Can't tell \u2014 observational estimate has wrong sign.",
     'Sensitivity Analysis'),
]

# Build table
responses = get_all_responses()

print(f'{"Notebook":<18s} {"Question":<45s} {"Your Answer":<30s} {"Correct Answer":<45s} {"Theorem":<25s}')
print('-' * 165)

for nb_id, q_id, nb_label, question, correct, theorem in trap_map:
    key = f"{nb_id}_{q_id}"
    if key in responses:
        student_ans = responses[key].get('response', 'N/A')
        if len(str(student_ans)) > 28:
            student_ans = str(student_ans)[:25] + '...'
    else:
        student_ans = 'Not attempted'
    
    correct_short = correct[:43] + '..' if len(correct) > 43 else correct
    print(f'{nb_label:<18s} {question[:43]:<45s} {str(student_ans):<30s} {correct_short:<45s} {theorem:<25s}')

print()
attempted = sum(1 for nb_id, q_id, *_ in trap_map if f"{nb_id}_{q_id}" in responses)
print(f'Notebooks completed: {attempted}/{len(trap_map)}')

In [ ]:
#@title Disaster Sidebar: Maimonides' Rule (Angrist & Lavy)

tab_contents = []

# Tab 1: The Story
story_out = widgets.Output()
with story_out:
    print("MAIMONIDES' RULE: A Regression Discontinuity Success")
    print('='*55)
    print()
    print('In Israel, a 12th-century rule by Maimonides caps class sizes at 40.')
    print('When enrollment hits 41, the school must split into two classes.')
    print()
    print('This creates a sharp discontinuity: a school with 40 students has')
    print('one class of 40. A school with 41 students has classes of ~20-21.')
    print('The students in a school of 40 vs. 41 are virtually identical \u2014')
    print('but they experience very different class sizes.')
    print()
    print('Angrist & Lavy (1999) used this natural experiment to estimate the')
    print('causal effect of class size on test scores. They found that smaller')
    print('classes significantly improved achievement \u2014 a result that was')
    print('credible precisely because the variation was quasi-random.')
    print()
    print('This is RDD at its best: a policy rule creates a clean experiment')
    print('that nature (or bureaucracy) runs for you.')

# Tab 2: The Math
math_out = widgets.Output()
with math_out:
    print('FORMAL FRAMEWORK: RDD Identification')
    print('='*50)
    print()
    print('At the cutoff c, if E[Y\u2080|X=x] is continuous:')
    print()
    print('  \u03c4_RDD = lim(x\u2192c+) E[Y|X=x] - lim(x\u2192c-) E[Y|X=x]')
    print('         = E[Y\u2081 - Y\u2080 | X=c]')
    print()
    print('Key assumptions:')
    print('  1. Continuity of potential outcomes at the cutoff')
    print('  2. No manipulation of the running variable')
    print('  3. Local randomization near the cutoff')
    print()
    print("For Maimonides' Rule:")
    print('  Running variable: enrollment count')
    print('  Cutoff: 40 students')
    print('  Treatment: small class (enrollment > 40 \u2192 split)')
    print('  Outcome: test scores')
    print()
    print('Class size = enrollment / int(1 + (enrollment - 1) / 40)')
    print('This function has sharp drops at 41, 81, 121, ...')

# Tab 3: The Simulation
sim_out = widgets.Output()
with sim_out:
    print("SIMULATION: Maimonides' Rule RDD")
    print('='*50)
    print()
    
    mc = MonteCarloEngine()
    
    def maimonides_dgp(seed, n=500):
        rng = np.random.default_rng(seed)
        # Enrollment between 20 and 60
        enrollment = rng.integers(25, 55, n)
        # Class size determined by Maimonides' rule
        class_size = enrollment / np.floor(1 + (enrollment - 1) / 40).astype(float)
        # Test scores: smaller class = better scores (true effect = -0.5 per student)
        scores = 80 - 0.5 * class_size + 0.1 * enrollment + rng.normal(0, 5, n)
        treatment = (enrollment > 40).astype(float)
        return pd.DataFrame({'enrollment': enrollment, 'class_size': class_size,
                             'scores': scores, 'treatment': treatment})
    
    def maimonides_estimator(df):
        near = df[(df['enrollment'] >= 35) & (df['enrollment'] <= 45)]
        if len(near[near['treatment']==1]) < 3 or len(near[near['treatment']==0]) < 3:
            return np.nan
        X = sm.add_constant(near[['enrollment', 'treatment']])
        return sm.OLS(near['scores'], X).fit().params['treatment']
    
    run_btn = widgets.Button(description='Run Simulation', button_style='primary')
    sim_plot_out = widgets.Output()
    
    def run_sim(btn):
        with sim_plot_out:
            clear_output(wait=True)
            estimates = mc.quick_run(maimonides_estimator, maimonides_dgp, n_reps=1000)
            estimates = estimates[~np.isnan(estimates)]
            
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.hist(estimates, bins=40, color=COLORS['ols'], edgecolor='white', alpha=0.8)
            ax.axvline(np.mean(estimates), color=COLORS['bias'], linewidth=2,
                       linestyle=':', label=f'Mean: {np.mean(estimates):.2f}')
            ax.set_xlabel('Estimated Class Size Effect', fontsize=11)
            ax.set_ylabel('Count', fontsize=11)
            ax.set_title("RDD Estimates: Maimonides' Rule", fontsize=12)
            ax.legend(fontsize=10)
            fig.tight_layout()
            plt.show()
            
            print(f'Mean RDD estimate: {np.mean(estimates):.2f}')
            print(f'The discontinuity captures the class size effect cleanly.')
    
    run_btn.on_click(run_sim)
    display(run_btn)
    display(sim_plot_out)

tabs = widgets.Tab(children=[story_out, math_out, sim_out])
tabs.set_title(0, 'The Story')
tabs.set_title(1, 'The Math')
tabs.set_title(2, 'The Simulation')
display(tabs)

---

## The Closing

You've now seen every way a regression can fail. Most practitioners never learn this. They run regressions that look convincing and draw conclusions that feel solid, without knowing what could go wrong.

You're different now. You know the failure modes. You know the diagnostics. You know the repairs and their limits. You know how to write an honest autopsy report.

**This doesn't mean you should stop using regression. It means you're finally ready to start.**

---

<div style='background: #1a1a2e; color: white; padding: 30px; border-radius: 10px; margin: 20px 0; text-align: center;'>
    <h2 style='color: #D4A017; margin-top: 0;'>Regression Autopsy: Complete</h2>
    <p style='font-size: 16px;'>Eight Ways Your Model Is Lying to You</p>
    <p style='font-style: italic; color: #aaa;'>"You will never blindly trust a regression again — and you'll know exactly when you should."</p>
</div>